# 03 - Feature Engineering

Este notebook implementa la extracción de características para la detección de phishing/smishing.

## Contenido
1. Características textuales (TF-IDF, n-gramas)
2. Características específicas de SMS
3. Características específicas de Email
4. Combinación de características
5. Análisis de correlación

In [ ]:
# Imports
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import SMSLoader, TextPreprocessor
from src.features import TextFeatureExtractor, SMSFeatureExtractor, EmailFeatureExtractor

%matplotlib inline

## 1. Carga de Datos

In [ ]:
# Cargar datos preprocesados o raw
sms_loader = SMSLoader(data_dir='../data')

try:
    df = sms_loader.load_processed('sms_preprocessed.csv')
except FileNotFoundError:
    try:
        df = sms_loader.load_uci_sms_spam()
    except:
        from src.data.sms_loader import create_sample_dataset
        df = create_sample_dataset()
    
    # Preprocesar
    preprocessor = TextPreprocessor(lowercase=True, replace_urls=True)
    df['text_clean_ml'] = preprocessor.preprocess_batch(df['body'])

print(f"Dataset: {len(df)} mensajes")

## 2. Características Textuales (TF-IDF)

In [ ]:
# Extractor TF-IDF
tfidf_extractor = TextFeatureExtractor(
    method='tfidf',
    max_features=5000,
    ngram_range=(1, 2),
    use_char_ngrams=True,
    char_max_features=2000
)

# Extraer características
text_col = 'text_clean_ml' if 'text_clean_ml' in df.columns else 'body'
X_tfidf = tfidf_extractor.fit_transform(df[text_col])

print(f"Shape características TF-IDF: {X_tfidf.shape}")

In [ ]:
# Características más importantes por IDF
top_features = tfidf_extractor.get_top_features(n=20)
print("Top 20 características por IDF (más comunes):")
for feat, score in top_features.items():
    print(f"  {feat}: {score:.4f}")

## 3. Características Específicas de SMS

In [ ]:
# Extractor de características SMS
sms_extractor = SMSFeatureExtractor()

# Extraer características para todos los mensajes
sms_features_df = sms_extractor.extract_features_batch(df['body'])

print(f"Características SMS extraídas: {sms_features_df.shape[1]}")
print(f"\nCaracterísticas disponibles:")
for col in sms_features_df.columns:
    print(f"  - {col}")

In [ ]:
# Añadir etiquetas y analizar
sms_features_df['label'] = df['label'].values

# Comparar medias por clase
print("Comparación de características por clase:")
comparison = sms_features_df.groupby('label').mean()
display(comparison.T)

In [ ]:
# Visualizar características más discriminativas
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

features_to_plot = ['url_count', 'urgency_count', 'has_shortened_url', 
                    'char_count', 'uppercase_ratio', 'total_suspicious_keywords']

for ax, feature in zip(axes.flatten(), features_to_plot):
    if feature in sms_features_df.columns:
        sms_features_df.boxplot(column=feature, by='label', ax=ax)
        ax.set_title(feature)
        ax.set_xticklabels(['Legítimo', 'Smishing'])

plt.suptitle('Distribución de Características por Clase', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/figures/feature_distribution.png', dpi=150)
plt.show()

## 4. Matriz de Correlación

In [ ]:
# Matriz de correlación de características
plt.figure(figsize=(14, 12))
corr_matrix = sms_features_df.corr()
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0)
plt.title('Matriz de Correlación de Características SMS')
plt.tight_layout()
plt.savefig('../reports/figures/correlation_matrix.png', dpi=150)
plt.show()

In [ ]:
# Correlación con la etiqueta
label_corr = sms_features_df.corr()['label'].drop('label').sort_values(key=abs, ascending=False)
print("Correlación con la etiqueta (ordenada por valor absoluto):")
for feat, corr in label_corr.head(15).items():
    print(f"  {feat}: {corr:.4f}")

## 5. Guardado de Características

In [ ]:
# Guardar características extraídas
sms_features_df.to_csv('../data/processed/sms_features.csv', index=False)
print("Características guardadas en data/processed/sms_features.csv")

# Guardar extractor TF-IDF
tfidf_extractor.save('../models/tfidf_vectorizer.pkl')
print("Extractor TF-IDF guardado en models/tfidf_vectorizer.pkl")

## 6. Resumen

### Características extraídas:
1. **TF-IDF**: Representación vectorial del texto
2. **Características SMS**: URLs, urgencia, longitud, patrones
3. **Características de caracteres**: Mayúsculas, signos, emojis

### Características más discriminativas:
- [Completar según resultados]

### Próximos pasos:
- Entrenamiento de modelos (notebook 04)